# Wedding Planner

This notebook applies advanced concepts from context and state, mcp and multi-agent systems to build a destination wedding planning assistant. The assistant will be a multi-agent system with a coordinating agent and three sub-agents: a travel agent, a venue agent and a DJ (playlist) agent.

- Travel agent: will utilize an external MCP server to find flights to and from a selected destination
- Venue agent: will use the context of the selected destination and Tavily API to search for venues
- DJ: will use context about music preferences and the destination to develop an appropriate playlist for the occasion

#### Design

Orchestrator:
- The orchestrator will be configured with state and memory to capture preferences and decisions about the destination, flight, venue and music. The agent's state will have the following variables:
  - Destination
  - Flight preferences
  - Flight
  - Venue preferences
  - Venue
  - Wedding size
  - Music preferences
- The orchestrator agent will pass these state variables to the sub-agents as context when calling them
- The orchestator will have the following tools
  - Call travel agent
  - Call dj agent
  - Update destination preferences
  - Update venue preferences
  - Update music preferences
  - Update flight preferences
  - Update flight
  - Update venue

Travel Agent:
- The travel agent will have the following tools:
  - Search for flights
  - Evaluate flights
- Travel agent will inherit context from the orchestator agent when calling tools

DJ Agent:
- The DJ agent will have the following tools:
  - Search for music
  - Create playlist
- DJ agent will inherit context from the orchestrator agent when calling tools

## Building the Agentic System

In [37]:
# import modules
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import HumanMessage

from dotenv import load_dotenv

from dataclasses import dataclass

Creating an orchestrator state class to track the relevant variables

In [ ]:
class orchestratorState(AgentState):
    destination: str
    wedding_date: str
    flight_pref: str
    flight: str
    venue_pref: str
    venue_name: str
    wedding_size: int
    music_pref: str


### Writing subagent tools

#### Using the Departi MCP Server for flights

In [48]:
# Setting up the client
departi_client = MultiServerMCPClient({
    "departi": {
      "transport": "http",
      "url": "https://mcp.departi.eu/v3"
    }
  }
)

# Grabbing tools and resources
flight_tools = await departi_client.get_tools()

flight_resources = await departi_client.get_resources()

# Creating the flight subagent
departi_agent = create_agent(
    model='claude-haiku-4-5',
    tools=flight_tools
)

In [45]:
departi_test = await departi_agent.ainvoke({"messages":[HumanMessage(
    content="""Can you find flight options between Chicago and Paris? 
    I am planning to depart on December 1st, 2026. I want to fly business class."""
)]})

In [46]:
departi_test["messages"]

[HumanMessage(content='Can you find flight options between Chicago and Paris? \n    I am planning to depart on December 1st, 2026. I want to fly business class.', additional_kwargs={}, response_metadata={}, id='bc8a4a8e-bafb-42e3-a597-944d561858cc'),
 AIMessage(content=[{'id': 'toolu_01HnWaAa67Ztz6Y4moSkZvRj', 'caller': {'type': 'direct'}, 'input': {'origin': 'Chicago', 'destination': 'Paris', 'departure_date': '2026-12-01', 'cabin_class': 'business'}, 'name': 'departi_search_transport', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeHm1ELmRf6XsbaFVytSv', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 3481, 'output_tokens': 117, 'output_tokens_details': None, 'server_to

Defining the subagent tools for performing actions and updating the orchestrator's state

In [ ]:
@tool
def read_destination(runtime: ToolRuntime):
    '''Reads the selected destination from the orchestrator's state'''
    try:
        return runtime.state["destination"]
    except:
        return "No destination could be found in state"

@tool
def read_wedding_size(runtime: ToolRuntime):
    '''Reads the planned size of the wedding from the orchestrator's state'''
    try:
        return runtime.state["wedding_size"]
    except:
        return "No wedding size could be found in state"

In [ ]:
# configure the agent